# D.3 · Hỏi thử adapter SFT trên Kaggle Private
Notebook **chỉ suy luận**. Gắn hai input riêng: **D3-code** đã giải nén và **Output của lần train liên tục** (thư mục có cả run_manifest.json và best_adapter/). Không cần D3-Release, USER_TEST, PDF, phiếu kiểm hoặc hồ sơ người thật. Bật T4 và Internet để tải mô hình nền công khai đúng revision; câu hỏi không gửi tới API. Chỉ nhập câu minh họa không có thông tin cá nhân; Kaggle có thể giữ output trong version. Đây không phải tư vấn pháp lý hay đánh giá D.5.

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, subprocess, sys, tempfile
from importlib import metadata

# Đường dẫn của notebook pilot đã chạy; nếu mount đổi, dùng Copy path rồi sửa tại đây.
RUN_DIR = Path('/kaggle/input/datasets/caoh48/output-version/d3-r8-a16-lr0.0001-e2-s41')
CODE_DIR = Path('/kaggle/input/datasets/caoh48/d3-code')
INSTALL_PACKAGES = True  # Chỉ khi thiếu pin và kế hoạch pip không thay stack GPU.
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
RELEASE_SHA = '73771db3ea89245114009d7bc0488966f83382606db16c2573545480ff97af74'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

In [ ]:
def read_json(p):
    return json.loads(p.read_text(encoding='utf-8'))

def digest(p):
    with p.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

adapter = RUN_DIR / 'best_adapter'
for p in [RUN_DIR / 'run_manifest.json', RUN_DIR / 'summary.json', adapter / 'selection.json', adapter / 'adapter_config.json', adapter / 'adapter_model.safetensors', CODE_DIR / 'src/d3_data.py', CODE_DIR / 'requirements-d3.txt']:
    if not p.is_file():
        raise FileNotFoundError(f'Thiếu {p}. Kiểm tra RUN_DIR và CODE_DIR bằng Copy path; không dùng thư mục epoch-0001.')
run = read_json(RUN_DIR / 'run_manifest.json')
summary = read_json(RUN_DIR / 'summary.json')
selection = read_json(adapter / 'selection.json')
if (run['model'] != {'id': MODEL_ID, 'revision': REVISION} or run['release']['manifest_sha256'] != RELEASE_SHA
        or run['config']['seed'] != 41 or summary['best_epoch'] != 2 or selection['epoch'] != 2
        or read_json(adapter / 'adapter_config.json')['base_model_name_or_path'] != MODEL_ID):
    raise ValueError('Không đúng run pilot release-01 / adapter chọn ở epoch 2.')
prompt_file = CODE_DIR / 'src/d3_data.py'
if digest(prompt_file) != 'c69592e3e1db834534f31321e2caf47555cea478ccfa2049dbb662844216822e' or digest(prompt_file) != run['source']['src/d3_data.py']:
    raise ValueError('Mã tạo prompt không khớp bản SFT đã chạy.')
SYSTEM_PROMPT = runpy.run_path(str(prompt_file))['SYSTEM_PROMPT']
print('Pilot seed 41 · best_adapter epoch 2 · SHA-256 trọng số:', digest(adapter / 'adapter_model.safetensors'))

In [ ]:
pins = [line for line in (CODE_DIR / 'requirements-d3.txt').read_text().splitlines() if line and not line.startswith('#')]
if set(pins) != {f'{k}=={v}' for k, v in run['runtime']['packages'].items()}:
    raise ValueError('Các phiên bản thư viện không khớp manifest train.')
def installed(pin):
    name, version = pin.split('==')
    try:
        return metadata.version(name) == version
    except metadata.PackageNotFoundError:
        return False
if not all(map(installed, pins)):
    if not INSTALL_PACKAGES:
        raise RuntimeError('Thiếu thư viện; bật INSTALL_PACKAGES và chạy lại ô này.')
    with tempfile.TemporaryDirectory() as tmp:
        constraint, report = Path(tmp)/'torch.txt', Path(tmp)/'plan.json'
        constraint.write_text('torch==' + metadata.version('torch'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-c', str(constraint), *pins]
        subprocess.run(cmd + ['--dry-run', '--report', str(report)], check=True)
        changes = [x['metadata']['name'].lower() for x in read_json(report)['install']]
        if any(n.startswith(('torch', 'triton', 'cuda', 'nvidia')) for n in changes):
            raise RuntimeError(f'Dừng để tránh thay stack GPU: {changes}')
        subprocess.run(cmd, check=True)
print('Thư viện D.3 đã sẵn sàng. Nếu kernel đã import bản cũ trước khi cài, Restart Session rồi chạy từ đầu.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU T4 trong Session options, sau đó chạy từ đầu.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False, use_fast=True)
if hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() != run['tokenizer']['chat_template_sha256']:
    raise ValueError('Chat template khác lần SFT.')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, quantization_config=quant, dtype=torch.float16, device_map={'': 0}, trust_remote_code=False, use_safetensors=True)
model = PeftModel.from_pretrained(base, str(adapter), is_trainable=False, local_files_only=True).eval()
print('Sẵn sàng hỏi thử:', MODEL_ID, '· adapter', model.active_adapter, '·', torch.cuda.get_device_name(0))

In [ ]:
@torch.inference_mode()
def hoi(cau_hoi):
    q = cau_hoi.strip()
    if not q or any(s in q for s in tokenizer.all_special_tokens):
        raise ValueError('Nhập một câu hỏi văn bản không rỗng, không chứa token điều khiển.')
    turns = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': q}]
    ids = tokenizer.apply_chat_template(turns, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    width = ids.shape[1]
    budget = min(160, run['config']['max_length'] - width)
    if budget < 1:
        raise ValueError('Câu quá dài cho cấu hình pilot; rút ngắn rồi hỏi lại.')
    output = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=budget, do_sample=False, num_beams=1, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    tail = output[0, width:]
    print(tokenizer.decode(tail, skip_special_tokens=True).strip() or '[Mô hình không sinh nội dung]')
    if tail.numel() == budget and tail[-1].item() != tokenizer.eos_token_id:
        print('[Dừng ở giới hạn token; câu trả lời có thể chưa hoàn chỉnh.]')

### Hỏi từng câu — chỉ chạy lại ô câu hỏi khi muốn đổi nội dung
Hai câu dưới đây là **minh họa kỹ thuật**, không phải khảo sát hay test độc lập. Mỗi lần hỏi là một lượt riêng, không mang lịch sử. Với câu trong phạm vi, mở phụ lục Quyết định 1852/QĐ-TTPVHCC, mục B.4, đối chiếu điều kiện, đơn vị thời gian và trích dẫn. Với câu ngoài phạm vi, xem mô hình có nêu giới hạn hay bịa thủ tục. Không coi câu có vẻ hợp lý là đã xác minh.

In [ ]:
hoi('Theo phương án của Hà Nội về xác nhận thông tin hộ tịch, thời hạn dự kiến khác thế nào giữa hồ sơ không cần xác minh và hồ sơ phải xác minh?')

In [ ]:
hoi('Tôi muốn xin thị thực du lịch Nhật Bản thì cần chuẩn bị những giấy tờ gì?')